# LLM Training Data Investigation

**Goal:** Diagnose why we're generating only 2.66M training examples instead of 4.2M.

**Key Issues to Investigate:**
1. Raw dataset completeness (80K items vs 137K expected)
2. Metadata extraction quality (description, features)
3. Type E co-purchase sparsity (683K vs 2.57M expected)

**Date:** 2025-11-20

## Setup

In [12]:
# Mount Google Drive for persistent storage
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [13]:
import os
import json
import gzip
import pickle
import numpy as np
import pandas as pd
from pathlib import Path
from collections import Counter, defaultdict
import matplotlib.pyplot as plt

# Set paths
WORK_DIR = Path('/content/drive/MyDrive/colab/tiger_semantic_id')
RQVAE_DIR = WORK_DIR / 'rq_vae_building' / 'artifacts'
DATA_DIR = RQVAE_DIR
ARTIFACTS_DIR = RQVAE_DIR
LLM_DIR = WORK_DIR / 'llm_finetuning'

print("=" * 60)
print("DIRECTORY CHECK")
print("=" * 60)
print(f"Data dir: {DATA_DIR}")
print(f"  Exists: {DATA_DIR.exists()}")
print(f"Artifacts dir: {ARTIFACTS_DIR}")
print(f"  Exists: {ARTIFACTS_DIR.exists()}")
print(f"LLM dir: {LLM_DIR}")
print(f"  Exists: {LLM_DIR.exists()}")
print("=" * 60)

DIRECTORY CHECK
Data dir: /content/drive/MyDrive/colab/tiger_semantic_id/rq_vae_building/artifacts
  Exists: True
Artifacts dir: /content/drive/MyDrive/colab/tiger_semantic_id/rq_vae_building/artifacts
  Exists: True
LLM dir: /content/drive/MyDrive/colab/tiger_semantic_id/llm_finetuning
  Exists: True


## Phase 1: Raw Dataset Verification

Check if we have the complete Amazon 2023 Video Games dataset.

In [21]:
print("=" * 60)
print("PHASE 1: RAW DATASET VERIFICATION")
print("=" * 60)

# Find the raw metadata file
meta_file = DATA_DIR / 'item_metadata.json'

print(f"\n[1.1] Checking raw metadata file...")
print(f"Path: {meta_file}")
print(f"Exists: {meta_file.exists()}")

if meta_file.exists():
    file_size_mb = meta_file.stat().st_size / (1024**2)
    print(f"File size: {file_size_mb:.1f} MB")

    # Load and count items in the JSON file
    print(f"\nLoading and counting items in JSON file (this may take a moment)...")
    with open(meta_file, 'rt', encoding='utf-8') as f:
        # Assuming item_metadata.json contains a single JSON object (e.g., a dictionary of items)
        raw_metadata_content = json.load(f)
    row_count = len(raw_metadata_content) # Count top-level items

    print(f"\n\n" + "=" * 60)
    print(f"RAW DATASET STATISTICS")
    print("=" * 60)
    print(f"Total items: {row_count:,}")
    print(f"Expected: ~80,840 (from LLM_DATA.md for extracted metadata)")
    print(f"Gap: {80840 - row_count:,} items ({100*(80840-row_count)/80840:.1f}%)")

    if row_count < 70000:
        print("\n⚠️  WARNING: Item count is suspiciously low!")
        print("   This suggests an issue with metadata extraction or the dataset used.")
    elif row_count < 80000:
        print("\n⚠️  Item count is lower than expected but not catastrophic.")
    else:
        print("\n✅ Item count looks good!")
    print("=" * 60)
else:
    print("\n❌ Raw metadata file not found!")
    print(f"   Expected location: {meta_file}")

PHASE 1: RAW DATASET VERIFICATION

[1.1] Checking raw metadata file...
Path: /content/drive/MyDrive/colab/tiger_semantic_id/rq_vae_building/artifacts/item_metadata.json
Exists: True
File size: 100.7 MB

Loading and counting items in JSON file (this may take a moment)...


RAW DATASET STATISTICS
Total items: 80,840
Expected: ~80,840 (from LLM_DATA.md for extracted metadata)
Gap: 0 items (0.0%)

✅ Item count looks good!


In [3]:
print("=" * 60)
print("[1.2] Checking item_metadata.json")
print("=" * 60)

# Check the extracted metadata
item_meta_file = ARTIFACTS_DIR / 'item_metadata.json'

if item_meta_file.exists():
    with open(item_meta_file, 'r') as f:
        item_metadata = json.load(f)

    print(f"\nExtracted items: {len(item_metadata):,}")
    print(f"Expected: ~80,840 (from LLM_DATA.md)")
    print(f"Reference: ~137,269")

    # Show extraction rate if we have raw count
    if 'row_count' in locals():
        extraction_rate = 100 * len(item_metadata) / row_count
        print(f"\nExtraction rate: {extraction_rate:.1f}% of raw dataset")
        print(f"Missing: {row_count - len(item_metadata):,} items")

    print("\n" + "=" * 60)
else:
    print("\n❌ item_metadata.json not found!")
    print(f"   Expected location: {item_meta_file}")

[1.2] Checking item_metadata.json

❌ item_metadata.json not found!
   Expected location: /Users/fox/Projects/CodexProjects/recsys_playground/tiger_semantic_id/artifacts/item_metadata.json


## Phase 2: Metadata Quality Analysis

Analyze the quality of extracted metadata (title, description, features).

In [4]:
print("=" * 60)
print("PHASE 2: METADATA QUALITY ANALYSIS")
print("=" * 60)

if 'item_metadata' in locals():
    # Analyze field availability
    field_stats = {
        'title': {'count': 0, 'lengths': []},
        'description': {'count': 0, 'lengths': []},
        'features': {'count': 0, 'lengths': []},
        'category_leaf': {'count': 0, 'lengths': []}
    }

    for item_id, meta in item_metadata.items():
        for field in field_stats.keys():
            if field in meta and meta[field]:
                field_stats[field]['count'] += 1
                field_stats[field]['lengths'].append(len(str(meta[field])))

    print("\n[2.1] Field Availability")
    print("-" * 60)
    total_items = len(item_metadata)

    for field, stats in field_stats.items():
        count = stats['count']
        pct = 100 * count / total_items if total_items > 0 else 0

        if stats['lengths']:
            avg_len = np.mean(stats['lengths'])
            min_len = np.min(stats['lengths'])
            max_len = np.max(stats['lengths'])
            print(f"{field:15s}: {count:6,} ({pct:5.1f}%) | avg_len={avg_len:6.1f} chars (min={min_len}, max={max_len})")
        else:
            print(f"{field:15s}: {count:6,} ({pct:5.1f}%) | NO DATA")

    # Check filtering requirements
    print("\n[2.2] Filtering Analysis")
    print("-" * 60)
    print("Reference filtering requirements:")
    print("  - Title length ≥ 20 characters")
    print("  - Description length ≥ 100 characters")
    print("  - Items must have BOTH to pass\n")

    title_valid = 0
    desc_valid = 0
    both_valid = 0

    for meta in item_metadata.values():
        has_title = 'title' in meta and len(str(meta['title'])) >= 20
        has_desc = 'description' in meta and len(str(meta['description'])) >= 100

        if has_title:
            title_valid += 1
        if has_desc:
            desc_valid += 1
        if has_title and has_desc:
            both_valid += 1

    print(f"Items with title ≥20 chars:       {title_valid:6,} ({100*title_valid/total_items:5.1f}%)")
    print(f"Items with description ≥100 chars: {desc_valid:6,} ({100*desc_valid/total_items:5.1f}%)")
    print(f"Items passing BOTH filters:        {both_valid:6,} ({100*both_valid/total_items:5.1f}%)")

    print(f"\nExpected after filtering: ~42,100 (from LLM_DATA.md)")
    print(f"Predicted after filtering: {both_valid:,}")
    print(f"Match: {'✅ YES' if abs(both_valid - 42100) < 1000 else '❌ NO'}")

else:
    print("\n⚠️  Skipping - item_metadata not loaded")

PHASE 2: METADATA QUALITY ANALYSIS

⚠️  Skipping - item_metadata not loaded


In [5]:
print("=" * 60)
print("[2.3] Sample Metadata Inspection")
print("=" * 60)

if 'item_metadata' in locals():
    # Show a few examples of items that FAIL filtering
    print("\nExamples of items that FAIL filtering (no description ≥100 chars):\n")

    fail_count = 0
    for item_id, meta in list(item_metadata.items())[:100]:
        has_desc = 'description' in meta and len(str(meta['description'])) >= 100

        if not has_desc and fail_count < 3:
            print(f"Item {item_id}:")
            print(f"  Title: {meta.get('title', 'N/A')[:80]}...")
            desc = meta.get('description', 'N/A')
            print(f"  Description: {str(desc)[:100]}...")
            print(f"  Description length: {len(str(desc)) if desc != 'N/A' else 0} chars")
            print(f"  Has features: {'features' in meta}")
            print()
            fail_count += 1

    # Show examples that PASS
    print("\nExamples of items that PASS filtering:\n")

    pass_count = 0
    for item_id, meta in item_metadata.items():
        has_title = 'title' in meta and len(str(meta['title'])) >= 20
        has_desc = 'description' in meta and len(str(meta['description'])) >= 100

        if has_title and has_desc and pass_count < 3:
            print(f"Item {item_id}:")
            print(f"  Title: {meta.get('title', 'N/A')[:80]}...")
            desc = meta.get('description', 'N/A')
            print(f"  Description: {str(desc)[:100]}...")
            print(f"  Description length: {len(str(desc))} chars")
            print(f"  Has features: {'features' in meta}")
            print()
            pass_count += 1
else:
    print("\n⚠️  Skipping - item_metadata not loaded")

[2.3] Sample Metadata Inspection

⚠️  Skipping - item_metadata not loaded


## Phase 3: Co-occurrence Analysis

Investigate why Type E is generating so few examples.

In [6]:
print("=" * 60)
print("PHASE 3: CO-OCCURRENCE ANALYSIS")
print("=" * 60)

# Load user sequences from LLM artifacts
user_seq_file = LLM_DIR / 'user_sequences.json'

if user_seq_file.exists():
    print(f"\n[3.1] Loading filtered user sequences...")
    with open(user_seq_file, 'r') as f:
        user_sequences = json.load(f)

    # Convert string keys to int
    user_sequences = {int(k): v for k, v in user_sequences.items()}

    print(f"Loaded sequences for {len(user_sequences):,} users")

    # Analyze sequence lengths
    seq_lengths = [len(seq) for seq in user_sequences.values()]

    print("\n[3.2] Sequence Length Distribution")
    print("-" * 60)
    print(f"Total sequences: {len(seq_lengths):,}")
    print(f"Total interactions: {sum(seq_lengths):,}")
    print(f"Average length: {np.mean(seq_lengths):.2f}")
    print(f"Median length: {np.median(seq_lengths):.1f}")
    print(f"Min length: {np.min(seq_lengths)}")
    print(f"Max length: {np.max(seq_lengths)}")
    print(f"\nPercentiles:")
    for p in [25, 50, 75, 90, 95, 99]:
        print(f"  p{p:2d}: {np.percentile(seq_lengths, p):.1f}")

    print(f"\nReference (from LLM_DATA.md):")
    print(f"  Average: 6.5")
    print(f"  Ours: {np.mean(seq_lengths):.2f}")
    print(f"  Gap: {100*(np.mean(seq_lengths) - 6.5)/6.5:+.1f}%")

else:
    print(f"\n❌ user_sequences.json not found at {user_seq_file}")

PHASE 3: CO-OCCURRENCE ANALYSIS

❌ user_sequences.json not found at /Users/fox/Projects/CodexProjects/recsys_playground/tiger_semantic_id/llm_artifacts/user_sequences.json


In [7]:
print("=" * 60)
print("[3.3] Co-occurrence Matrix Analysis")
print("=" * 60)

if 'user_sequences' in locals():
    # Build co-occurrence matrix
    print("\nBuilding co-occurrence matrix (window-based)...")

    cooccurrence = defaultdict(lambda: defaultdict(int))
    window_size = 5  # Same as reference implementation

    for user_id, sequence in user_sequences.items():
        for i, item_i in enumerate(sequence):
            # Look at items within window
            for j in range(max(0, i - window_size), min(len(sequence), i + window_size + 1)):
                if i != j:
                    item_j = sequence[j]
                    cooccurrence[item_i][item_j] += 1

    # Analyze co-occurrence density
    cooccurrence_counts = [len(items) for items in cooccurrence.values()]

    print(f"\nItems with co-occurrences: {len(cooccurrence):,}")
    print(f"Average co-occurrences per item: {np.mean(cooccurrence_counts):.1f}")
    print(f"Median co-occurrences per item: {np.median(cooccurrence_counts):.1f}")
    print(f"Min: {np.min(cooccurrence_counts)}")
    print(f"Max: {np.max(cooccurrence_counts)}")

    print(f"\nPercentiles:")
    for p in [25, 50, 75, 90, 95]:
        print(f"  p{p:2d}: {np.percentile(cooccurrence_counts, p):.1f}")

    # Estimate Type E examples
    total_cooccurrences = sum(cooccurrence_counts)
    print(f"\n" + "=" * 60)
    print(f"TYPE E ESTIMATION")
    print("=" * 60)
    print(f"Total co-occurrences: {total_cooccurrences:,}")
    print(f"\nWith exhaustive mode (all co-occurrences):")
    print(f"  Expected Type E examples: ~{total_cooccurrences:,}")
    print(f"\nActual Type E (from LLM_DATA.md): 683,301")
    print(f"Expected Type E (reference): 2,569,685")
    print(f"\nOur estimate vs actual: {100*total_cooccurrences/683301:.1f}%")
    print(f"Our estimate vs reference: {100*total_cooccurrences/2569685:.1f}%")

    # Check if this matches our current Type E count
    if abs(total_cooccurrences - 683301) < 50000:
        print(f"\n✅ Co-occurrence count matches current Type E generation")
        print(f"   → Type E generation is working correctly")
        print(f"   → Problem is sparse co-occurrences due to fewer/shorter sequences")
    else:
        print(f"\n⚠️  Co-occurrence count doesn't match Type E")
        print(f"   → May be an issue with Type E generation logic")
else:
    print("\n⚠️  Skipping - user_sequences not loaded")

[3.3] Co-occurrence Matrix Analysis

⚠️  Skipping - user_sequences not loaded


In [8]:
print("=" * 60)
print("[3.4] Sequence Sparsity Visualization")
print("=" * 60)

if 'seq_lengths' in locals() and 'cooccurrence_counts' in locals():
    # Plot distributions
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Sequence length distribution
    axes[0].hist(seq_lengths, bins=50, edgecolor='black', alpha=0.7)
    axes[0].axvline(np.mean(seq_lengths), color='red', linestyle='--', label=f'Mean: {np.mean(seq_lengths):.1f}')
    axes[0].axvline(6.5, color='green', linestyle='--', label='Reference: 6.5')
    axes[0].set_xlabel('Sequence Length')
    axes[0].set_ylabel('Count')
    axes[0].set_title('User Sequence Length Distribution')
    axes[0].legend()
    axes[0].grid(alpha=0.3)

    # Co-occurrence distribution
    axes[1].hist(cooccurrence_counts, bins=50, edgecolor='black', alpha=0.7)
    axes[1].axvline(np.mean(cooccurrence_counts), color='red', linestyle='--',
                    label=f'Mean: {np.mean(cooccurrence_counts):.1f}')
    axes[1].set_xlabel('Co-occurrences per Item')
    axes[1].set_ylabel('Count')
    axes[1].set_title('Co-occurrence Distribution')
    axes[1].legend()
    axes[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(WORK_DIR / 'notebooks' / 'tiger_semantic_id' / 'sequence_analysis.png', dpi=100, bbox_inches='tight')
    print("\n✅ Saved visualization to sequence_analysis.png")
    plt.show()
else:
    print("\n⚠️  Skipping - data not available")

[3.4] Sequence Sparsity Visualization

⚠️  Skipping - data not available


## Phase 4: Root Cause Summary & Recommendations

In [9]:
print("=" * 60)
print("PHASE 4: ROOT CAUSE SUMMARY & RECOMMENDATIONS")
print("=" * 60)

print("\n📊 FINDINGS SUMMARY\n")

# Summarize findings
findings = []

if 'row_count' in locals():
    if row_count < 100000:
        findings.append({
            'severity': '🔴 CRITICAL',
            'issue': 'Raw dataset has fewer than 100K items',
            'impact': 'This is the PRIMARY cause of missing examples',
            'recommendation': 'Re-download the complete Amazon 2023 Video Games dataset'
        })
    elif row_count < 130000:
        findings.append({
            'severity': '🟡 MODERATE',
            'issue': f'Raw dataset has {row_count:,} items (expected ~137K)',
            'impact': 'Contributing to lower example count',
            'recommendation': 'Verify dataset download URL and completeness'
        })

if 'item_metadata' in locals():
    if len(item_metadata) < 70000:
        findings.append({
            'severity': '🔴 CRITICAL',
            'issue': f'Only {len(item_metadata):,} items extracted from raw dataset',
            'impact': 'Losing items during metadata extraction',
            'recommendation': 'Check metadata extraction logic in data preparation'
        })

if 'field_stats' in locals():
    desc_coverage = field_stats['description']['count'] / len(item_metadata) if item_metadata else 0
    if desc_coverage < 0.5:
        findings.append({
            'severity': '🟡 MODERATE',
            'issue': f'Only {100*desc_coverage:.1f}% of items have descriptions',
            'impact': 'Many items fail filtering (need desc ≥100 chars)',
            'recommendation': 'Check description extraction from Amazon 2023 format (list joining)'
        })

if 'total_cooccurrences' in locals():
    if total_cooccurrences < 1000000:
        findings.append({
            'severity': '🟡 MODERATE',
            'issue': f'Only {total_cooccurrences:,} co-occurrences (expected ~2.6M)',
            'impact': 'Type E will be severely undersized',
            'recommendation': 'Sequences are too sparse - need more/longer sequences'
        })

# Print findings
for i, finding in enumerate(findings, 1):
    print(f"{i}. {finding['severity']} {finding['issue']}")
    print(f"   Impact: {finding['impact']}")
    print(f"   → {finding['recommendation']}")
    print()

if not findings:
    print("✅ No critical issues found! Data looks good.\n")

# Final recommendations
print("\n" + "=" * 60)
print("RECOMMENDED ACTIONS (Priority Order)")
print("=" * 60)
print("\n1. ⭐ Check raw dataset completeness")
print("   - Verify you downloaded the FULL Video Games dataset")
print("   - URL: https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw/meta_categories/meta_Video_Games.jsonl.gz")
print("   - Expected: ~137K items")

print("\n2. Check metadata extraction")
print("   - Ensure load_meta_df() processes all rows")
print("   - Verify description list joining works correctly")
print("   - Check for silent failures/exceptions")

print("\n3. If data is truly smaller, consider:")
print("   - Relaxing filters (title ≥10, desc ≥50) as temporary workaround")
print("   - Accepting lower example count (~2.7M instead of 4.2M)")
print("   - Using a different dataset (e.g., All_Beauty, Sports)")

print("\n" + "=" * 60)
print("INVESTIGATION COMPLETE")
print("=" * 60)

PHASE 4: ROOT CAUSE SUMMARY & RECOMMENDATIONS

📊 FINDINGS SUMMARY

✅ No critical issues found! Data looks good.


RECOMMENDED ACTIONS (Priority Order)

1. ⭐ Check raw dataset completeness
   - Verify you downloaded the FULL Video Games dataset
   - URL: https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw/meta_categories/meta_Video_Games.jsonl.gz
   - Expected: ~137K items

2. Check metadata extraction
   - Ensure load_meta_df() processes all rows
   - Verify description list joining works correctly
   - Check for silent failures/exceptions

3. If data is truly smaller, consider:
   - Relaxing filters (title ≥10, desc ≥50) as temporary workaround
   - Accepting lower example count (~2.7M instead of 4.2M)
   - Using a different dataset (e.g., All_Beauty, Sports)

INVESTIGATION COMPLETE
